# CS336 Spring 2026 - 11 Scaled Dot-Product / Causal Attention
----
> 10 已经准备好 RMSNorm、SwiGLU 和 Residual。本章聚焦 Transformer 的核心交互机制：Attention。

$$
\operatorname{Attention}(Q,K,V)
=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.
$$

注意这次主要的中间 tensor 是 `(B,H,T,T)`：每个 query 位置对所有 key 位置给出一个 attention weight。


## 1. 从 `(B,T,D)` 得到 `(B,H,T,D_head)`

使用 `H` 个 attention heads，令 `D_head=D/H`（必须整除）。

```text
x (B,T,D)
    ↓ Q/K/V projection
q,k,v (B,T,D)
    ↓ reshape
(B,T,H,D_head)
    ↓ transpose
(B,H,T,D_head)
```

每个 head 独立计算 attention，最后把 heads 拼回 `(B,T,D)`。


In [1]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)

B, T, D, H = 2, 5, 8, 2
D_head = D // H
x = torch.randn(B, T, D)
qkv = nn.Linear(D, 3 * D, bias=False)
q, k, v = qkv(x).chunk(3, dim=-1)


def split_heads(z, num_heads):
    batch_size, seq_len, model_dim = z.shape
    assert model_dim % num_heads == 0
    return z.reshape(batch_size, seq_len, num_heads, model_dim // num_heads).transpose(
        1, 2
    )


Q, K, V = (split_heads(t, H) for t in (q, k, v))

assert Q.shape == K.shape == V.shape == (B, H, T, D_head)

print("Q/K/V:", tuple(Q.shape))

Q/K/V: (2, 2, 5, 4)


## 2. Scaled Dot-Product

$$
S_{b,h,i,j}=\frac{\langle Q_{b,h,i,:},K_{b,h,j,:}\rangle}{\sqrt{D_{\mathrm{head}}}}.
$$

为什么除以 `sqrt(D_head)`？如果 Q、K 的分量方差合适且近似独立，dot product 的方差会随 `D_head` 增长；缩放可避免 softmax 过早饱和。

```text
Q: (B,H,T,D_head)
K^T: (B,H,D_head,T)
scores: (B,H,T,T)
```


In [2]:
scores = Q @ K.transpose(-2, -1) / math.sqrt(D_head)
assert scores.shape == (B, H, T, T)
print("scores:", tuple(scores.shape))


scores: (2, 2, 5, 5)


## 3. Causal mask：位置 i 不能看未来 j>i

语言模型预测下一个 token 时，不允许当前位置看到未来 token。通常构造一个上三角 mask，将未来位置得分设为 `-inf`，再做 softmax：

$$
P_{i,j}=0,\quad j>i.
$$

重要顺序是 **mask → softmax**。不要在 softmax 后简单把未来位置变成 0 而不归一化。


In [3]:
def causal_scaled_dot_product_attention(Q, K, V, *, return_weights=False):
    d_head = Q.shape[-1]
    t_query, t_key = Q.shape[-2], K.shape[-2]
    assert t_query == t_key, "此教学例子只讨论 self-attention"
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_head)
    future = torch.ones((t_query, t_key), dtype=torch.bool, device=Q.device).triu(1)
    scores = scores.masked_fill(future, float("-inf"))
    probabilities = torch.softmax(scores, dim=-1)
    output = probabilities @ V
    return (output, probabilities) if return_weights else output


attention_output, weights = causal_scaled_dot_product_attention(
    Q, K, V, return_weights=True
)
assert attention_output.shape == (B, H, T, D_head)
assert weights.shape == (B, H, T, T)
for i in range(T):
    assert torch.all(weights[..., i, i + 1 :] == 0)
torch.testing.assert_close(weights.sum(dim=-1), torch.ones(B, H, T))
print("causal mask 后的 attention weights（第一个 head）:")
print(weights[0, 0].detach().round(decimals=3))


causal mask 后的 attention weights（第一个 head）:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6010, 0.3990, 0.0000, 0.0000, 0.0000],
        [0.3370, 0.3620, 0.3010, 0.0000, 0.0000],
        [0.2350, 0.3200, 0.1950, 0.2500, 0.0000],
        [0.1960, 0.2990, 0.1740, 0.1780, 0.1530]])


## 4. 参照 PyTorch SDPA 做数值比较

`torch.nn.functional.scaled_dot_product_attention` 提供了参考接口。设置 `is_causal=True`、`dropout_p=0.0` 与前面的教学实现比较。

注意 PyTorch 可能根据硬件选择不同 kernel；浮点舍入误差不必完全按 bit 相同。


In [4]:
reference_output = F.scaled_dot_product_attention(
    Q, K, V, is_causal=True, dropout_p=0.0
)
torch.testing.assert_close(attention_output, reference_output, atol=1e-6, rtol=1e-5)
print("手写 attention 与 torch SDPA：数值一致")


手写 attention 与 torch SDPA：数值一致


## 5. 完整 Multi-Head Attention 模块

`QKV` 可以合并为一次矩阵投影，然后拆分三个 Tensor。Attention 输出先从 `(B,H,T,D_head)` 转回 `(B,T,D)`，最后进行 output projection。

在后面的 12 中还会把 RoPE 施加到 Q 和 K 上；本章先确认**不含位置编码**的 causal MHA 是正确的。


In [5]:
class TeachingCausalMHA(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = (split_heads(tensor, self.n_heads) for tensor in (q, k, v))
        z = causal_scaled_dot_product_attention(q, k, v)
        z = z.transpose(1, 2).contiguous().reshape(b, t, d)
        return self.out(z)


mha = TeachingCausalMHA(D, H)
y = mha(x)
assert y.shape == x.shape
assert sum(p.numel() for p in mha.parameters()) == 4 * D * D
print("MHA:", tuple(x.shape), "→", tuple(y.shape))


MHA: (2, 5, 8) → (2, 5, 8)


## 6. 用“修改未来 token”检验因果性

检查 `attn_mask` 的形状并不够：若某一步 transpose 搞错，结果仍可能偷偷泄漏未来信息。

用两个几乎相同的输入：前半部分完全一致，后半部分故意大幅修改。如果是 causal attention，前面的输出必须保持不变。


In [6]:
x1 = torch.randn(2, 7, D)
x2 = x1.clone()
x2[:, 4:] = torch.randn_like(x2[:, 4:]) * 10.0
y1, y2 = mha(x1), mha(x2)
torch.testing.assert_close(y1[:, :4], y2[:, :4], atol=1e-6, rtol=1e-6)
print("修改未来输入，前缀输出保持相同：通过")


修改未来输入，前缀输出保持相同：通过


## 7. 梯度、单 token 与 shape 边界

序列长度为 1 时，attention weights 必须为 1（只有自己）。任意合法 `(B,T,D)` 输入的输出仍是 `(B,T,D)`，所有参数应可参与梯度反传。

当前朴素 attention 的注意力矩阵需要 $O(BHT^2)$ 规模的中间存储，因此序列变长后会很昂贵；Assignment 2 才深入系统优化。


In [7]:
for t in (1, 2, 8):
    z = torch.randn(3, t, D, requires_grad=True)
    result = mha(z)
    assert result.shape == z.shape
    result.square().mean().backward()
    assert torch.isfinite(z.grad).all()

q1 = torch.randn(1, H, 1, D_head)
k1 = torch.randn_like(q1)
v1 = torch.randn_like(q1)
o1, p1 = causal_scaled_dot_product_attention(q1, k1, v1, return_weights=True)
torch.testing.assert_close(p1, torch.ones_like(p1))
torch.testing.assert_close(o1, v1)
print("T=1、多个 T、forward/backward：通过")


T=1、多个 T、forward/backward：通过


## 8. 小结 → 12 RoPE / Transformer Block / LM

Causal MHA 现在能正确计算：

```text
(B,T,D) → Q/K/V → (B,H,T,D_head)
                  ↓ causal attention
                (B,H,T,D_head)
                  ↓ concatenate + projection
                (B,T,D)
```

下一章加入 RoPE，连接 10 的 Pre-Norm + SwiGLU，并组成具有 next-token prediction 能力的 Transformer LM。
